# StyleGAN2-ADA — BRATS2013 Brain MRI

Pure-PyTorch StyleGAN2 + Adaptive Discriminator Augmentation (ADA), no custom CUDA/HIP kernels.
Auto-selects device: `cuda` → DirectML (AMD) → MPS (Apple) → CPU.

This notebook drives the `stylegan2_ada/` package: **preprocess → visualize → train → generate**.

> **Data reality:** BRATS2013 test+leaderboard = **35 subjects** (all high-grade glioma, no labels),
> sliced into ~17.5k 2D images. With only 35 brains the model mostly memorizes/interpolates them —
> good for learning the pipeline and augmentation experiments, not a diverse tumor generator.

Design doc: `docs/superpowers/specs/2026-07-02-stylegan2-ada-brats-design.md`

## 0. Setup

In [ ]:
# Make the local stylegan2_ada package importable from the notebook.
import sys, os
sys.path.insert(0, os.path.abspath('.'))

# One-time deps (uncomment if needed):
# %pip install SimpleITK scipy
# On the AMD/Windows box also: %pip install torch-directml

import torch
from stylegan2_ada.device import pick_device, probe, describe

device = pick_device()          # or pick_device('mps'|'cuda'|'dml'|'cpu')
caps = probe(device)
print(describe(caps))

## 1. Preprocess: `.mha` 3D volumes → 2D grayscale PNG slices

Run once. Skips automatically if the slices already exist. Unzip the BRATS archives into
`data/brats_raw/` first (Challenge + Leaderboard).

In [ ]:
from pathlib import Path

RAW = 'data/brats_raw'
SLICES = 'data/brats_slices'
RESOLUTION = 128

if not list(Path(SLICES).glob('*.png')):
    !PYTHONPATH=. {sys.executable} -m stylegan2_ada.data.prepare_brats --raw {RAW} --out {SLICES} --resolution {RESOLUTION}
else:
    print(f'{len(list(Path(SLICES).glob("*.png")))} slices already present in {SLICES}')

## 2. Peek at real slices

In [ ]:
import matplotlib.pyplot as plt
from stylegan2_ada.data.dataset import SliceDataset

ds = SliceDataset(SLICES, resolution=RESOLUTION, xflip=True)
print('dataset size (with xflip):', len(ds))

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax in axes.flat:
    img = ds[torch.randint(len(ds), ()).item()][0]        # (H,W) in [-1,1]
    ax.imshow((img + 1) / 2, cmap='gray'); ax.axis('off')
plt.suptitle('Real BRATS slices (all modalities pooled)'); plt.tight_layout(); plt.show()

## 3. Configure & train

Edit `Config` below. For a quick demo keep `kimg` small; for a real run bump it to a few thousand.
`train()` writes samples to `runs/brats/samples/` and checkpoints to `runs/brats/checkpoints/`.

> On MPS, ADA drops geometric augments (grid_sample backward unimplemented) — handled automatically.
> On DirectML, keep `amp=False`. The startup probe prints what's enabled.

In [ ]:
from stylegan2_ada.train import Config, train

cfg = Config(
    data=SLICES,
    out='runs/brats',
    resolution=RESOLUTION,
    batch=16,
    kimg=200,                 # <- demo length; use ~5000 for a real run
    sample_every_kimg=20,
    ckpt_every_kimg=100,
    workers=0,                # 0 is safest inside notebooks
    # device=None -> auto; set 'cpu' for a very quick test
)
train(cfg)

## 4. Generate from a checkpoint

In [ ]:
from stylegan2_ada.generate import load_generator, sample
from torchvision.utils import make_grid

ckpt = 'runs/brats/checkpoints/final.pt'
G, gcfg = load_generator(ckpt, device)
imgs = sample(G, gcfg, device, num=16, psi=0.7)            # (16,1,H,W) in [0,1]

grid = make_grid(imgs, nrow=4).cpu().permute(1, 2, 0)[..., 0]
plt.figure(figsize=(8, 8)); plt.imshow(grid, cmap='gray'); plt.axis('off')
plt.title('Generated slices (G_ema, truncation 0.7)'); plt.show()

## 5. Latent (W-space) interpolation

In [ ]:
from stylegan2_ada.generate import interpolate

seq = interpolate(G, gcfg, device, steps=8, psi=0.7)       # (8,1,H,W)
grid = make_grid(seq, nrow=8).cpu().permute(1, 2, 0)[..., 0]
plt.figure(figsize=(16, 2)); plt.imshow(grid, cmap='gray'); plt.axis('off')
plt.title('W-space interpolation'); plt.show()

## 6. (Optional) FID

Needs `scipy` and network access (downloads InceptionV3 weights once).

In [ ]:
# from stylegan2_ada.metrics.fid import compute_fid
# fid = compute_fid(G, ds, gcfg if isinstance(gcfg, dict) else cfg, device, num=2000)
# print('FID:', round(fid, 2))